# Python File Handling

#

## 1- File opening (the open() function and modes)
Short explanation: open() returns a file object that lets you read from or write to a file. Modes control whether the file is opened for reading ('r'), writing ('w'), appending ('a'), exclusive creation ('x'), and whether the file is treated as text ('t') or binary ('b'). Always prefer the with statement (context manager) which closes the file automatically.

Common modes

'r' — read (default). File must exist.

'w' — write (truncate if exists or create new).

'a' — append (create if not exists).

'x' — exclusive creation; fails if file exists.

'rb', 'wb', 'ab' — binary modes.

'r+', 'w+' — read/write.

Examples:

In [ ]:
# simple open & close
f = open('notes.txt', 'r', encoding='utf-8')
text = f.read()
f.close()


# recommended: using with (context manager)
with open('notes.txt', 'r', encoding='utf-8') as f:
    text = f.read()
# file is closed here automatically


# open for writing (creates or truncates)
with open('output.txt', 'w', encoding='utf-8') as f:
    f.write('Hello, world\n')


# open in binary mode
with open('image.png', 'rb') as f:
    data = f.read()

When you open a file using open(), Python gives you a file object that you can read from or write to. However, if you open the file manually, you are responsible for closing it with f.close() when you are done. If you forget to close the file, it can lead to problems such as memory/resource leaks, locked files, or incomplete writes.

Using a with statement is recommended because it automatically opens and closes the file for you. When the with block finishes (even if an error occurs), Python guarantees that the file will be closed safely. This makes your code cleaner, safer, and more reliable.

#

## 2- Reading files
Short explanation: Common reading methods:

read() — read entire file (useful for small files).

read(size) — read up to size bytes/characters.

readline() — read one line at a time.

readlines() — return a list of all lines (memory-heavy for large files).

Iterating over the file object — memory-efficient, recommended for line-by-line processing.

Examples:

In [ ]:
# read entire file
with open('data.txt', 'r', encoding='utf-8') as f:
    content = f.read()


# read line by line with readline()
with open('data.txt', 'r', encoding='utf-8') as f:
    line = f.readline()
    while line:
        print(line.strip())
        line = f.readline()


# iterate directly (preferred for lines)
with open('data.txt', 'r', encoding='utf-8') as f:
    for line in f:
        print(line.strip())   # replaced process(line)


#

## 3- Writing and creating files
Short explanation: Writing modes let you create new files and write data. Use 'w' to create/overwrite, 'a' to append, and 'x' to create exclusively. Prefer with to ensure the file is closed and data flushed.

Examples:

In [ ]:
# overwrite or create
with open('log.txt', 'w', encoding='utf-8') as f:
    f.write('Start of log\n')


# append
with open('log.txt', 'a', encoding='utf-8') as f:
    f.write('New event happened\n')


# write multiple lines
lines = ['first line\n', 'second line\n']
with open('lines.txt', 'w', encoding='utf-8') as f:
    f.writelines(lines)


# using print to write with automatic newline
with open('out.txt', 'w', encoding='utf-8') as f:
    print('a line', file=f)

#

## 4- Deleting files (and directories)
Short explanation: Use the os module or pathlib to remove files. For directories, use os.rmdir() for empty directories and shutil.rmtree() for non-empty directories.

Examples:

In [ ]:
import os
from pathlib import Path
import shutil


# remove a single file (os)
if os.path.exists('temp.txt'):
    os.remove('temp.txt')


# using pathlib
p = Path('temp.txt')
if p.exists():
    p.unlink()


# remove empty directory
os.rmdir('my_empty_dir')


# remove directory tree (be careful)
shutil.rmtree('old_project')

#

## 5- Working with CSV files
Short explanation: CSV (Comma-Separated Values) is a common plain-text format for tabular data. Use Python's built-in csv module for reliable parsing and writing, because it handles quoting, escapes, and delimiters properly. When opening CSV files for writing, pass newline='' to open() to avoid blank lines on Windows.

Examples:

In [ ]:
import csv


# reading CSV (row as list)
with open('students.csv', 'r', encoding='utf-8', newline='') as f:
    reader = csv.reader(f)
    for row in reader:
        print('Name:', row[0], 'Score:', row[1])


# writing CSV
rows = [
['name', 'score'],
['Alice', '85'],
['Bob', '92'],
]

with open('out.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(rows)


# DictReader / DictWriter (useful for header-based access)
with open('students.csv', 'r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader: # each row is an OrderedDict / dict
        print(row['name'], row['score'])


with open('out_dict.csv', 'w', encoding='utf-8', newline='') as f:
    fieldnames = ['name', 'score']
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow({'name': 'Carol', 'score': 88})

#

## 6- Working with JSON files
Short explanation: JSON is a common text format for structured data (mappings and lists). Use the json module: json.load() to read from a file and json.dump() to write Python objects to a file. Use indent= for human-readable formatting.

Examples:

In [ ]:
import json


# read JSON
with open('data.json', 'r', encoding='utf-8') as f:
    data = json.load(f) # data can be dict, list, etc.


# write JSON
payload = {'name': 'Alice', 'scores': [85, 90]}
with open('payload.json', 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)


# update JSON safely (read-modify-write)
from pathlib import Path
p = Path('records.json')
if p.exists():
    data = json.loads(p.read_text(encoding='utf-8'))
else:
    data = []


# modify data
data.append({'id': 3, 'name': 'New'})

# atomic-ish write: write to temp file then replace
p_tmp = p.with_suffix('.tmp')
p_tmp.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')
p_tmp.replace(p)

#

## Quick cheatsheet

| Category | Common Operations / Syntax |
|----------|-----------------------------|
| **Opening** | `open(path, mode='r', encoding='utf-8', newline=None)` |
| **Read** | `f.read()` • `f.readline()` • `for line in f:` |
| **Write** | `f.write(text)` • `f.writelines(list_of_lines)` • `print(..., file=f)` |
| **Delete** | `os.remove(path)` • `Path(path).unlink()` • `shutil.rmtree(path)` |
| **CSV** | `csv.reader` • `csv.writer` • `csv.DictReader` • `csv.DictWriter` *(open with `newline=''`)* |
| **JSON** | `json.load(f)` • `json.dump(obj, f, indent=2)` |

